# **Google drive mount**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **All library import**

In [ ]:
!pip install -q gensim kaggle

import os, re, math, multiprocessing
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.models.callbacks import CallbackAny2Vec
from gensim.utils import simple_preprocess
print('Setup complete.')

Setup complete.


# **Quora Dataset Download from kaggle**

In [ ]:
os.environ['KAGGLE_USERNAME'] = 'msbithi '
os.environ['KAGGLE_KEY']      = 'KGAT_248779a3b7805e99c5e99a86400c191d'

!kaggle datasets download -d quora/question-pairs-dataset -q
!unzip -o question-pairs-dataset.zip -d . > /dev/null

df_quora  = pd.read_csv('questions.csv', encoding='utf-8', on_bad_lines='skip')
questions = []
for _, row in df_quora.iterrows():
    if pd.notna(row['question1']): questions.append(str(row['question1']))
    if pd.notna(row['question2']): questions.append(str(row['question2']))
print(f'Total questions for training: {len(questions)}')

# Tokenizer
def tokenize(text):
    words = re.findall(r"[a-z']+", text.lower())
    return [w.strip("'") for w in words if w.strip("'")]

Dataset URL: https://www.kaggle.com/datasets/quora/question-pairs-dataset
License(s): other
Total questions for training: 808699


# **Doc2Vec Train Use Quora Dataset**

In [ ]:
tagged_docs = [TaggedDocument(words=tokenize(q), tags=[i])
               for i, q in enumerate(questions)]

class EpochLogger(CallbackAny2Vec):
    def __init__(self): self.epoch = 0
    def on_epoch_end(self, model):
        self.epoch += 1
        print(f'  Epoch {self.epoch}/100 complete', flush=True)

# GitHub parameters: window=15, dbow_words=1, sample=1e-5
model_quora = Doc2Vec(
    vector_size = 300,
    window      = 15,
    min_count   = 1,
    workers     = multiprocessing.cpu_count(),
    epochs      = 100,
    dm          = 0,
    dbow_words  = 1,
    negative    = 5,
    sample      = 1e-5,
    hs          = 0
)
model_quora.build_vocab(tagged_docs)
print(f'Vocabulary size: {len(model_quora.wv)}')
print('Training Doc2Vec model (~3 hours)...')

model_quora.train(tagged_docs,
                  total_examples=model_quora.corpus_count,
                  epochs=model_quora.epochs,
                  callbacks=[EpochLogger()])
print('Training complete!')

model_quora.save('/content/drive/MyDrive/doc2vec_quora_v2.model')
print('Model saved!')


# **Loading Voice Command Fingerprinting Dataset (Kennedy et al., 2019) - 1,000 traffic traces, 100 voice commands from Amazon Echo**

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/Dataset/trace_csv'

def get_label_from_filename(filename):
    name = filename.replace('.csv', '')
    name = re.sub(r'_5_[\d_]+(?:_capture)?_\d+$', '', name)
    name = re.sub(r'_5_\d+s_L_\d+$', '', name)
    name = re.sub(r'_5_L_\d+$', '', name)
    name = re.sub(r'_5_\d+s_\d+$', '', name)
    name = re.sub(r'_5_\d+s$', '', name)
    name = re.sub(r'_5_\d+$', '', name)
    name = re.sub(r'_\d+$', '', name)

    return name

def load_trace(filepath):
    df = pd.read_csv(filepath)
    return list(zip(df['size'].tolist(), df['direction'].tolist()))

def load_dataset(dataset_path):
    traces, labels = [], []
    all_files = sorted([f for f in os.listdir(dataset_path) if f.endswith('.csv')])
    for filename in all_files:
        label = get_label_from_filename(filename)
        trace = load_trace(os.path.join(dataset_path, filename))
        traces.append(trace)
        labels.append(label)
    unique_labels = sorted(set(labels))
    label_to_id = {l: i for i, l in enumerate(unique_labels)}
    label_ids = np.array([label_to_id[l] for l in labels])
    counts = defaultdict(int)
    for l in labels:
        counts[l] += 1
    vals = list(counts.values())
    print(f'Total traces: {len(traces)}, Unique commands: {len(unique_labels)}')
    print(f'Traces per command — min: {min(vals)}, max: {max(vals)}')
    return traces, labels, label_ids, label_to_id, unique_labels

traces, labels, label_ids, label_to_id, unique_labels = load_dataset(DATASET_PATH)

Total traces: 1000, Unique commands: 100
Traces per command — min: 10, max: 10


# **Utility Functions for Feature Extraction and Bucket/Histogram Generation for Traffic Features**

In [ ]:
def getSectionList(start, end, interval):
    rangeList = list(range(start, end, interval))
    sectionList = [0] * len(rangeList)
    return rangeList, sectionList

def computeRange(rangeList, feat):
    for i in range(len(rangeList) - 1):
        if rangeList[i] <= feat < rangeList[i+1]:
            return i
    if feat >= rangeList[-1]:
        return len(rangeList) - 1
    return 0

print('Bucket functions defined!')

Bucket functions defined!


# **All feature extraction functions defined for 4 attack algorithms**
  - LL-Jaccard: Set-based features
  - LL-NB: 60-dim packet size histogram
  - VNG++: 403-dim burst-based features
  - P-SVM: 405-dim combined features

In [ ]:
def trace_to_jaccard_set(trace):
    tmpSet = set()
    for size, direction in trace:
        elem = int(size) * int(direction)
        tmpSet.add(elem)
    return tmpSet

def build_consensus_set(list_of_sets):
    n = len(list_of_sets)
    threshold = math.ceil(n / 2)
    all_elems = set()
    for s in list_of_sets:
        all_elems = all_elems.union(s)
    final_set = set()
    for elem in all_elems:
        count = sum(1 for s in list_of_sets if elem in s)
        if count >= threshold:
            final_set.add(elem)
    return final_set

def jaccard_similarity(set_x, set_y):
    inter = len(set_x & set_y)
    union = len(set_x | set_y)
    return inter / union if union > 0 else 0.0

def trace_to_bayes_feature(trace, range_start=-1500, range_end=1501, interval=50):
    rangeList, sectionList = getSectionList(range_start, range_end, interval)
    for size, direction in trace:
        feat = int(size) * int(direction)
        idx = computeRange(rangeList, feat)
        sectionList[idx] += 1
    return np.array(sectionList, dtype=float)

def calculate_bursts(trace):
    if not trace:
        return []
    burstList = []
    direction = int(trace[0][1])
    tmp_burst = int(trace[0][0]) * direction
    for size, direc in trace[1:]:
        direc = int(direc)
        if direction != direc:
            burstList.append(tmp_burst)
            direction = direc
            tmp_burst = int(size) * direc
        else:
            tmp_burst += int(size) * direc
    burstList.append(tmp_burst)
    return burstList

def trace_to_vngpp_feature(trace, range_start=-400000, range_end=400001, interval=2000):
    UpStreamTotal = sum(int(s) for s, d in trace if int(d) == 1)
    DownStreamTotal = sum(int(s) for s, d in trace if int(d) == -1)
    TotalTraceTime = len(trace)

    burstList = calculate_bursts(trace)
    rangeList, sectionList = getSectionList(range_start, range_end, interval)
    for feat in burstList:
        idx = computeRange(rangeList, feat)
        sectionList[idx] += 1

    feature = [TotalTraceTime, UpStreamTotal, DownStreamTotal]
    feature.extend(sectionList)
    return np.array(feature, dtype=float)

def trace_to_svm_feature(trace, range_start=-200000, range_end=200001, interval=1000):
    sizes   = [int(s) for s, d in trace]
    direcs  = [int(d) for s, d in trace]
    UpPackNum    = sum(1 for d in direcs if d == 1)
    DownPackNum  = sum(1 for d in direcs if d == -1)
    UpStreamTotal   = sum(s for s, d in zip(sizes, direcs) if d == 1)
    DownStreamTotal = sum(s for s, d in zip(sizes, direcs) if d == -1)
    packNum = len(trace)
    inPackRatio = DownPackNum / packNum if packNum > 0 else 0
    unique_sizes = len(set(sizes))

    burstList = calculate_bursts(trace)
    burstNum = len(burstList)

    rangeList, sectionList = getSectionList(range_start, range_end, interval)
    for feat in burstList:
        idx = computeRange(rangeList, feat)
        sectionList[idx] += 1

    feature = [UpStreamTotal, DownStreamTotal, inPackRatio, packNum, burstNum]
    feature.extend(sectionList)
    return np.array(feature, dtype=float)

print('All feature extraction functions defined!')

All feature extraction functions defined!


# **BuFLO countermeasure implemented (Dyer et al., 2012)**
  Parameters: d=1000, rho=50, tau=20
  Effect: 548% overhead, 330% delay, accuracy ↓ to 5-14%

In [ ]:
def apply_buflo(trace, d=1000, rho=50, tau=20):
    padded = [(d, int(direction)) for (size, direction) in trace]
    total_needed = int(tau * rho)
    while len(padded) < total_needed:
        padded.append((d, 1))
    return padded

print('BuFLO defined!')

BuFLO defined!


# **Load the pre-trained Quora Doc2Vec model, Preprocess Tokenizer, command vector generate & sanity check**

In [ ]:
from gensim.models.doc2vec import Doc2Vec
from sklearn.metrics.pairwise import cosine_similarity

model_quora = Doc2Vec.load('/content/drive/MyDrive/doc2vec_quora_v2.model')
if not hasattr(model_quora, 'syn1neg'):
    model_quora.syn1neg = np.zeros((len(model_quora.wv), model_quora.vector_size))
    print('Fixed syn1neg!')
print(f'Model loaded | vocab: {len(model_quora.wv)} | vector_size: {model_quora.vector_size}')

def tokenize(text):
    words = re.findall(r"[a-z']+", text.lower())
    return [w.strip("'") for w in words if w.strip("'")]

def get_command_vector(text, model, alpha=0.025, epochs=50):
    words = tokenize(text.replace('_', ' '))
    return model.infer_vector(words, alpha=alpha, epochs=epochs)

command_vectors_quora = {}
for cmd in unique_labels:
    words = tokenize(cmd.replace('_', ' '))
    command_vectors_quora[cmd] = model_quora.infer_vector(words, alpha=0.025, epochs=50)
print(f'Semantic vectors ready for {len(command_vectors_quora)} voice commands.')

# Update sanity check
v1 = get_command_vector('what is the weather', model_quora)
v2 = get_command_vector('what is the weather tomorrow', model_quora)
v3 = get_command_vector('set a timer for thirty seconds', model_quora)

print(f'\nSimilarity check:')
print(f'  weather vs weather tomorrow : {cosine_similarity(v1.reshape(1,-1), v2.reshape(1,-1))[0][0]:.3f}')
print(f'  weather vs set a timer      : {cosine_similarity(v1.reshape(1,-1), v3.reshape(1,-1))[0][0]:.3f}')



Model loaded | vocab: 82632 | vector_size: 300
Semantic vectors ready for 100 voice commands.

Similarity check:
  weather vs weather tomorrow : 0.801
  weather vs set a timer      : 0.438


# **Semantic Distance (SD) & Normalized Semantic Distance (NSD) Metrics Implementation**

In [ ]:
def semantic_distance(vec1, vec2):
    v1 = np.array(vec1, dtype=float).reshape(1, -1)
    v2 = np.array(vec2, dtype=float).reshape(1, -1)
    return float(np.clip(cosine_similarity(v1, v2)[0][0], -1.0, 1.0))

def normalized_semantic_distance(predicted_cmd, actual_cmd, all_commands, cmd_vectors):
    pred_vec = cmd_vectors[predicted_cmd]

    distances = []
    for cmd in all_commands:
        dist = semantic_distance(pred_vec, cmd_vectors[cmd])
        distances.append((cmd, dist))

    distances.sort(key=lambda x: x[1], reverse=True)

    actual_dist = semantic_distance(pred_vec, cmd_vectors[actual_cmd])
    for rank, (cmd, dist) in enumerate(distances):
        if cmd == actual_cmd:
            return rank

    return len(all_commands)

print('Semantic distance functions defined.')

Semantic distance functions defined.


# **Main Attack Pipeline: 5-Fold Cross-Validation for All Four Attacks**
*   Running 5-fold cross-validation (800 train + 200 test per fold)
*   Implementing 4 attacks: LL-Jaccard, LL-NB, VNG++, P-SVM (AdaBoost)
*   Calculating 3 metrics: Accuracy, SD (correct predictions only), NSD (Algorithm 1)
*   Supporting BuFLO obfuscation (for Table II experiments)
*   Averaging results across 5 folds to generate Tables I & II



In [ ]:
def run_all_attacks_with_both_sd(traces, label_ids, labels,
                                 obfuscate_buflo=False,
                                 buflo_d=1000, buflo_rho=50, buflo_tau=20,
                                 cmd_vectors=None,
                                 model=None):

    id_to_label = {i: l for i, l in enumerate(unique_labels)}

    working = traces
    if obfuscate_buflo:
        working = [apply_buflo(t, buflo_d, buflo_rho, buflo_tau) for t in traces]
        print(f'BuFLO: d={buflo_d}, rho={buflo_rho}, tau={buflo_tau}')

    # Pre-compute features
    sets_jac  = [trace_to_jaccard_set(t)    for t in working]
    feats_nb  = np.array([trace_to_bayes_feature(t) for t in working])
    feats_vng = np.array([trace_to_vngpp_feature(t) for t in working])
    feats_svm = np.array([trace_to_svm_feature(t)   for t in working])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {
        'LL-Jaccard':       {'acc': [], 'sd_correct': [], 'sd_all': [], 'nsd': []},
        'LL-NB':            {'acc': [], 'sd_correct': [], 'sd_all': [], 'nsd': []},
        'VNG++':            {'acc': [], 'sd_correct': [], 'sd_all': [], 'nsd': []},
        'P-SVM (AdaBoost)': {'acc': [], 'sd_correct': [], 'sd_all': [], 'nsd': []}
    }

    for fold, (tr_idx, te_idx) in enumerate(
            skf.split(np.zeros(len(label_ids)), label_ids), 1):
        print(f'  Fold {fold}/5 ...', end=' ')
        y_tr = label_ids[tr_idx]
        y_te = label_ids[te_idx]

        # --- LL-Jaccard ---
        class_sets = defaultdict(list)
        for i in tr_idx:
            class_sets[label_ids[i]].append(sets_jac[i])
        class_model = {cls: build_consensus_set(sl) for cls, sl in class_sets.items()}
        preds_jac = []
        for i in te_idx:
            best_cls, best_score = -1, -1
            for cls, cset in class_model.items():
                score = jaccard_similarity(sets_jac[i], cset)
                if score > best_score:
                    best_score, best_cls = score, cls
            preds_jac.append(best_cls)

        # --- LL-NB ---
        nb = GaussianNB()
        nb.fit(feats_nb[tr_idx], y_tr)
        preds_nb = nb.predict(feats_nb[te_idx])

        # --- VNG++ ---
        vng = GaussianNB()
        vng.fit(feats_vng[tr_idx], y_tr)
        preds_vng = vng.predict(feats_vng[te_idx])

        # --- P-SVM (AdaBoost) ---
        from sklearn.tree import DecisionTreeClassifier
        dt = DecisionTreeClassifier(max_depth=5)
        ada = AdaBoostClassifier(dt, n_estimators=500,
                                  learning_rate=0.1, random_state=42)
        ada.fit(feats_svm[tr_idx], y_tr)
        preds_ada = ada.predict(feats_svm[te_idx])

        algo_preds = {
            'LL-Jaccard':       preds_jac,
            'LL-NB':            preds_nb,
            'VNG++':            preds_vng,
            'P-SVM (AdaBoost)': preds_ada
        }

        for algo, preds in algo_preds.items():
            acc = accuracy_score(y_te, preds)
            sd_correct_list = []
            sd_all_list = []
            nsd_list = []

            if cmd_vectors is not None and model is not None:
                for actual_id, pred_id in zip(y_te, preds):
                    actual_cmd = id_to_label[actual_id]
                    pred_cmd = id_to_label[pred_id]

                    nsd = normalized_semantic_distance(pred_cmd, actual_cmd,
                                                       unique_labels, cmd_vectors)
                    nsd_list.append(nsd)

                    actual_vec = get_command_vector(actual_cmd, model, alpha=0.025, epochs=50)
                    pred_vec = get_command_vector(pred_cmd, model, alpha=0.025, epochs=50)
                    sd = semantic_distance(pred_vec, actual_vec)
                    sd_all_list.append(sd)

                    if actual_id == pred_id:
                        sd_correct_list.append(sd)

            results[algo]['acc'].append(acc)
            results[algo]['sd_correct'].append(np.mean(sd_correct_list) if sd_correct_list else 0)
            results[algo]['sd_all'].append(np.mean(sd_all_list) if sd_all_list else 0)
            results[algo]['nsd'].append(np.mean(nsd_list) if nsd_list else 0)

        print('done')

    final = {}
    for algo in results:
        final[algo] = {
            'acc': np.mean(results[algo]['acc']) * 100,
            'sd_correct': np.mean(results[algo]['sd_correct']),
            'sd_all': np.mean(results[algo]['sd_all']),
            'nsd': np.mean(results[algo]['nsd'])
        }
    return final

# **Table I: Voice Command Fingerprinting Results (No Obfuscation)**

In [ ]:
# Run with both SD values
print('Running Table I with both SD calculations...')
r1 = run_all_attacks_with_both_sd(traces, label_ids, labels,
                                  obfuscate_buflo=False,
                                  cmd_vectors=command_vectors_quora,
                                  model=model_quora)

# Paper values
paper_acc = {'LL-Jaccard': 17.4, 'LL-NB': 33.8, 'VNG++': 24.9, 'P-SVM (AdaBoost)': 33.4}
paper_sd = {'LL-Jaccard': 0.914, 'LL-NB': 0.924, 'VNG++': 0.916, 'P-SVM (AdaBoost)': 0.928}
paper_nsd = {'LL-Jaccard': 45.21, 'LL-NB': 33.25, 'VNG++': 42.67, 'P-SVM (AdaBoost)': 38.06}

print('\n' + '='*100)
print('TABLE I: Voice Command Fingerprinting — No Obfuscation')
print('='*100)
print(f'{"Algorithm":<22} {"Acc(Ours)":>8} {"Acc(Paper)":>8} {"SD-Correct":>10} {"SD(Paper)":>9} {"SD-All":>8} {"NSD(Ours)":>8} {"NSD(Paper)":>8}')
print('-'*100)
for a in ['LL-Jaccard', 'LL-NB', 'VNG++', 'P-SVM (AdaBoost)']:
    r = r1[a]
    print(f'{a:<22} {r["acc"]:>8.1f}%  {paper_acc[a]:>8.1f}%  {r["sd_correct"]:>10.3f}  {paper_sd[a]:>9.3f}  {r["sd_all"]:>8.3f}  {r["nsd"]:>8.2f}  {paper_nsd[a]:>8.2f}')
print('-'*100)
print(f'{"Random Guess":<22} {"1.0%":>8} {"1.0%":>8} {"N/A":>10} {"N/A":>9} {"N/A":>8} {"49.5":>8} {"49.5":>8}')
print('\nNote: SD-Correct shows values for correct predictions only (matching paper methodology).')
print('      SD-All shows average over all predictions (correct + wrong).')

Running Table I with both SD calculations...
  Fold 1/5 ... done
  Fold 2/5 ... done
  Fold 3/5 ... done
  Fold 4/5 ... done
  Fold 5/5 ... done

TABLE I: Voice Command Fingerprinting — No Obfuscation
Algorithm              Acc(Ours) Acc(Paper) SD-Correct SD(Paper)   SD-All NSD(Ours) NSD(Paper)
----------------------------------------------------------------------------------------------------
LL-Jaccard                 17.6%      17.4%       0.929      0.914     0.520     38.42     45.21
LL-NB                      28.5%      33.8%       0.932      0.924     0.571     32.92     33.25
VNG++                      22.6%      24.9%       0.931      0.916     0.550     35.11     42.67
P-SVM (AdaBoost)           28.3%      33.4%       0.934      0.928     0.579     32.08     38.06
----------------------------------------------------------------------------------------------------
Random Guess               1.0%     1.0%        N/A       N/A      N/A     49.5     49.5

Note: SD-Correct shows v

# **Validation: Semantic Distance Distribution Analysis for LL-NB**

In [ ]:
print("\n" + "="*60)
print("DETAILED ANALYSIS FOR LL-NB")
print("="*60)

def analyze_sd_distribution_detailed(traces, label_ids, labels, model, cmd_vectors):

    id_to_label = {i: l for i, l in enumerate(unique_labels)}

    from sklearn.model_selection import train_test_split
    tr_idx, te_idx = train_test_split(range(len(label_ids)),
                                      test_size=0.2,
                                      stratify=label_ids,
                                      random_state=42)

    # Pre-compute features
    sets_jac = [trace_to_jaccard_set(t) for t in traces]
    feats_nb = np.array([trace_to_bayes_feature(t) for t in traces])

    # Train LL-NB
    nb = GaussianNB()
    nb.fit(feats_nb[tr_idx], label_ids[tr_idx])
    preds = nb.predict(feats_nb[te_idx])

    # Analyze
    sd_correct = []
    sd_wrong = []

    print("\nAnalyzing 200 test samples (one fold)...")
    for actual_id, pred_id in zip(label_ids[te_idx], preds):
        actual_cmd = id_to_label[actual_id]
        pred_cmd = id_to_label[pred_id]

        # Get fresh vectors
        actual_vec = get_command_vector(actual_cmd, model)
        pred_vec = get_command_vector(pred_cmd, model)
        sd = semantic_distance(pred_vec, actual_vec)

        if actual_id == pred_id:
            sd_correct.append(sd)
        else:
            sd_wrong.append(sd)

    print(f"Correct predictions: {len(sd_correct)} ({len(sd_correct)/len(preds)*100:.1f}%)")
    print(f"Wrong predictions: {len(sd_wrong)} ({len(sd_wrong)/len(preds)*100:.1f}%)")
    print(f"\nSD for correct predictions: mean={np.mean(sd_correct):.4f}, std={np.std(sd_correct):.4f}")
    print(f"SD for wrong predictions: mean={np.mean(sd_wrong):.4f}, std={np.std(sd_wrong):.4f}")
    print(f"Overall SD: {np.mean(sd_correct + sd_wrong):.4f}")

    # Show distribution
    if len(sd_correct) > 0:
        print(f"\nCorrect SD range: min={np.min(sd_correct):.4f}, max={np.max(sd_correct):.4f}")
    if len(sd_wrong) > 0:
        print(f"Wrong SD range: min={np.min(sd_wrong):.4f}, max={np.max(sd_wrong):.4f}")

    return sd_correct, sd_wrong

# Run the analysis
sd_correct, sd_wrong = analyze_sd_distribution_detailed(
    traces, label_ids, labels, model_quora, command_vectors_quora
)


DETAILED ANALYSIS FOR LL-NB

Analyzing 200 test samples (one fold)...
Correct predictions: 54 (27.0%)
Wrong predictions: 146 (73.0%)

SD for correct predictions: mean=0.9917, std=0.0048
SD for wrong predictions: mean=0.3835, std=0.0901
Overall SD: 0.5477

Correct SD range: min=0.9709, max=0.9958
Wrong SD range: min=0.1936, max=0.6826


# **Table II: Voice Command Fingerprinting Results with BuFLO Obfuscation**

In [ ]:
print('Running Table II (BuFLO obfuscation) with both SD calculations...')
r2 = run_all_attacks_with_both_sd(traces, label_ids, labels,
                                  obfuscate_buflo=True,
                                  buflo_d=1000, buflo_rho=50, buflo_tau=20,
                                  cmd_vectors=command_vectors_quora,
                                  model=model_quora)

paper_acc2 = {'LL-Jaccard': 5.0,  'LL-NB': 10.5, 'VNG++': 8.3,  'P-SVM (AdaBoost)': 10.1}
paper_nsd2 = {'LL-Jaccard': 47.12, 'LL-NB': 44.33, 'VNG++': 46.05, 'P-SVM (AdaBoost)': 44.88}
algos = ['LL-Jaccard', 'LL-NB', 'VNG++', 'P-SVM (AdaBoost)']

print('\n' + '='*80)
print('TABLE II: Voice Command Fingerprinting — BuFLO Obfuscation')
print('='*80)
print(f'{"Algorithm":<22} {"Acc(Ours)":>10} {"Acc(Paper)":>10} {"NSD(Ours)":>10} {"NSD(Paper)":>10} {"SD-Correct":>12} {"SD-All":>8}')
print('-'*80)
for a in algos:
    r = r2[a]
    print(f'{a:<22} {r["acc"]:>9.1f}%  {paper_acc2[a]:>9.1f}%  {r["nsd"]:>10.2f}  {paper_nsd2[a]:>10.2f}  {r["sd_correct"]:>12.3f}  {r["sd_all"]:>8.3f}')
print('-'*80)
print(f'{"Random Guess":<22} {"1.0%":>9} {"1.0%":>9} {"49.5":>10} {"49.5":>10} {"N/A":>12} {"N/A":>8}')
print('\nNote: Paper only reports Accuracy and Normalized SD for Table II.')
print('      SD-Correct and SD-All shown here for completeness.')

Running Table II (BuFLO obfuscation) with both SD calculations...
BuFLO: d=1000, rho=50, tau=20
  Fold 1/5 ... done
  Fold 2/5 ... done
  Fold 3/5 ... done
  Fold 4/5 ... done
  Fold 5/5 ... done

TABLE II: Voice Command Fingerprinting — BuFLO Obfuscation
Algorithm               Acc(Ours) Acc(Paper)  NSD(Ours) NSD(Paper)   SD-Correct   SD-All
--------------------------------------------------------------------------------
LL-Jaccard                   1.0%        5.0%       49.50       47.12         0.949     0.485
LL-NB                        5.9%       10.5%       46.76       44.33         0.913     0.463
VNG++                        8.6%        8.3%       45.68       46.05         0.917     0.466
P-SVM (AdaBoost)             8.3%       10.1%       44.52       44.88         0.931     0.469
--------------------------------------------------------------------------------
Random Guess                1.0%      1.0%       49.5       49.5          N/A      N/A

Note: Paper only reports Accu